# Inference (COUGHVID)

從 `002_training_coughvid` 輸出的 test split 中隨機抽一筆進行推論，確保抽到的樣本不在訓練資料內。

## Imports

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch
from utils import preproces, CoughNet

## 選擇推論檔案

從 test split 隨機抽一筆（執行前請先跑完 `002_training_coughvid`）

也可以手動指定路徑，例如：
```python
fn_wav = 'data/coughvid_wav/你的檔案.wav'
```

In [6]:
COUGHVID_ROOT = 'C:/Users/aint/Desktop/RNN/Final_project/coughvid_wav'
TEST_CSV      = 'data/prepared_test_split.csv'

df_test = pd.read_csv(TEST_CSV)
sample  = df_test.sample(1).iloc[0]

fn_wav = os.path.join(COUGHVID_ROOT, sample['filename'])
print(f'檔案:     {sample["filename"]}')
print(f'真實標籤: {sample["label"]}')

檔案:     f028405b-2f4f-4379-ba3e-acc7a69f90be.wav
真實標籤: covid


## Inference

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

loaded_checkpoint = torch.load(
    'checkpoints/checkpoint_coughvid.pth',
    weights_only=False,
    map_location=device
)

hparams = loaded_checkpoint['hparams']
scaler  = loaded_checkpoint['scaler']
encoder = loaded_checkpoint['encoder']

model = CoughNet(len(hparams['features']))
model.eval()
model.load_state_dict(loaded_checkpoint['model_state'])

# 提取特徵
feature_row = preproces(fn_wav)
if isinstance(feature_row, pd.Series):
    feature_row = feature_row.to_dict()

df_features = pd.DataFrame([feature_row])
X = np.array(df_features[hparams['features']], dtype=np.float32)
X = torch.Tensor(scaler.transform(X))

outputs     = torch.softmax(model(X), 1)
predictions = torch.argmax(outputs.data, 1)

# 顯示結果
probs = outputs[0].detach().numpy()
predicted_class = encoder.classes_[predictions]

print(f'\n真實標籤: {sample["label"]}')
print(f'預測標籤: {predicted_class}')
print(f'結果:     {"✓ 正確" if predicted_class == sample["label"] else "✗ 錯誤"}')
print()
print('各類別機率:')
for i, cls in enumerate(encoder.classes_):
    bar = '█' * int(probs[i] * 30)
    print(f'  {cls:<15} {probs[i]:.4f}  {bar}')


真實標籤: covid
預測標籤: covid
結果:     ✓ 正確

各類別機率:
  covid           0.9647  ████████████████████████████
  healthy         0.0341  █
  symptomatic     0.0012  
